# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
# TODO
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.info()

(1338, 7)
['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


## A.2. Missing values & Duplicate data

In [3]:
# TODO
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'missing': missing, 'percent': missing_percent})
print(missing_df[missing_df['missing'] > 0].sort_values('missing', ascending=False))
print(df.duplicated().sum())

Empty DataFrame
Columns: [missing, percent]
Index: []
1


## A.3. Invalid values

In [4]:
# TODO
print((df['bmi'] <= 0).sum())
print((df['charges'] < 0).sum())
print((df['age'] <= 0).sum())
print(df['sex'].unique())
print(df['smoker'].unique())
print(df['region'].unique())

0
0
0
<StringArray>
['female', 'male']
Length: 2, dtype: str
<StringArray>
['yes', 'no']
Length: 2, dtype: str
<StringArray>
['southwest', 'southeast', 'northwest', 'northeast']
Length: 4, dtype: str


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [5]:
# TODO
df['bmi_group'] = pd.cut(df['bmi'], bins=[0, 25, 30, np.inf], labels=['Normal', 'Overweight', 'Obese'])
df[['bmi', 'bmi_group']].head()

,bmi,bmi_group
0,27.900,Overweight
1,33.770,Obese
2,33.000,Obese
3,22.705,Normal
4,28.880,Overweight


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
# TODO
print(df[['age', 'bmi', 'charges']].mean())
print(df[['age', 'bmi', 'charges']].median())
print(df[['age', 'bmi', 'charges']].mode().iloc[0])

age           39.207025
bmi           30.663397
charges    13270.422265
dtype: float64
age          39.000
bmi          30.400
charges    9382.033
dtype: float64
age          18.0000
bmi          32.3000
charges    1639.5631
Name: 0, dtype: float64


## Group 2 — Dispersion

In [7]:
# TODO
print(df[['age', 'bmi', 'charges']].std())
print(df[['age', 'bmi', 'charges']].var())
print(df['charges'].max() - df['charges'].min())
print(df[['age', 'bmi', 'charges']].quantile(0.75) - df[['age', 'bmi', 'charges']].quantile(0.25))  

age           14.049960
bmi            6.098187
charges    12110.011237
dtype: float64
age        1.974014e+02
bmi        3.718788e+01
charges    1.466524e+08
dtype: float64
62648.554110000005
age           24.000000
bmi            8.397500
charges    11899.625365
dtype: float64


## Group 3 — Location and Shape

In [8]:
# TODO
print(df[['age', 'bmi', 'charges']].quantile([0.25, 0.5, 0.75]))
print(df[['age', 'bmi', 'charges']].skew())
print(df[['age', 'bmi', 'charges']].kurt())

       age       bmi       charges
0.25  27.0  26.29625   4740.287150
0.50  39.0  30.40000   9382.033000
0.75  51.0  34.69375  16639.912515
age        0.055673
bmi        0.284047
charges    1.515880
dtype: float64
age       -1.245088
bmi       -0.050732
charges    1.606299
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [9]:
# TODO
charges_by_smoker = df.groupby('smoker')['charges'].mean()
print(charges_by_smoker)
print(charges_by_smoker['yes'] / charges_by_smoker['no'])
print(df.groupby(['region', 'smoker'])['charges'].mean())

smoker
no      8434.268298
yes    32050.231832
Name: charges, dtype: float64
3.800001458298319
region     smoker
northeast  no         9165.531672
           yes       29673.536473
northwest  no         8556.463715
           yes       30192.003182
southeast  no         8032.216309
           yes       34844.996824
southwest  no         8019.284513
           yes       32269.063494
Name: charges, dtype: float64


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [10]:
# TODO
print(df[df['smoker'] == 'yes'][['bmi', 'charges']].corr())
print(df[df['smoker'] == 'no'][['bmi', 'charges']].corr())

              bmi   charges
bmi      1.000000  0.806481
charges  0.806481  1.000000
              bmi   charges
bmi      1.000000  0.084037
charges  0.084037  1.000000


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [11]:
# TODO
charges_by_region = df.groupby('region')['charges'].mean()
print(charges_by_region)
print(charges_by_region.idxmax())

region
northeast    13406.384516
northwest    12417.575374
southeast    14735.411438
southwest    12346.937377
Name: charges, dtype: float64
southeast


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [12]:
# TODO
print(df.groupby('children')['charges'].mean())
print(df[['children', 'charges']].corr())

children
0    12365.975602
1    12731.171832
2    15073.563734
3    15355.318367
4    13850.656311
5     8786.035247
Name: charges, dtype: float64
          children   charges
children  1.000000  0.067998
charges   0.067998  1.000000


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [13]:
# TODO
print(df[['age', 'charges']].corr())

              age   charges
age      1.000000  0.299008
charges  0.299008  1.000000


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

1. Hút thuốc là yếu tố ảnh hưởng mạnh nhất đến chi phí bảo hiểm: người hút thuốc trả chi phí trung bình cao hơn nhiều lần so với người không hút.
2. BMI có tương quan dương với chi phí, và mối tương quan này rõ rệt hơn ở nhóm người hút thuốc — cho thấy hút thuốc kết hợp BMI cao làm tăng rủi ro sức khỏe/chi phí đáng kể.
3. Chi phí bảo hiểm có sự khác biệt giữa các vùng, nhưng chênh lệch không quá lớn so với ảnh hưởng của hút thuốc và BMI.
4. Số lượng con cái có ảnh hưởng không đáng kể (hoặc nhẹ) đến chi phí bảo hiểm.
5. Tuổi có tương quan dương với chi phí, phản ánh rủi ro sức khỏe tăng theo độ tuổi.